# HCMC Real Estate Price Intelligence — Colab end-to-end

[![Mở bằng Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haminhthong/hcmc-real-estate-price-intelligence/blob/main/notebooks/03_colab_end_to_end.ipynb)

Notebook chạy trực tiếp pipeline trong `src/`, không sao chép lại logic huấn luyện. Vì vậy kết quả giữa repository và Colab dùng chung một nguồn mã, dữ liệu mẫu, seed và phiên bản thư viện.

**Kết quả hiện tại:** 443 train, 93 validation, 86 calibration và 101 test; metrics được đọc từ `reports/metrics.json` sau lần chạy pipeline gần nhất.

## 1. Chuẩn bị repository và môi trường

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/haminhthong/hcmc-real-estate-price-intelligence.git"
REPO_NAME = "hcmc-real-estate-price-intelligence"
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules

if IN_COLAB:
    workspace = Path("/content")
    project_root = workspace / REPO_NAME
    if not project_root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(project_root)], check=True)
else:
    candidates = [Path.cwd(), Path.cwd().parent]
    project_root = next(
        (path.resolve() for path in candidates if (path / "src" / "pipeline.py").exists()),
        None,
    )
    if project_root is None:
        raise FileNotFoundError("Hãy chạy notebook từ thư mục gốc hoặc thư mục notebooks của dự án.")

os.chdir(project_root)
print(f"Thư mục dự án: {project_root}")

In [ ]:
# Colab cần đúng các phiên bản đã dùng để tạo artifact chuẩn.
if IN_COLAB:
    packages = [
        "pandas==2.2.3",
        "numpy==2.1.3",
        "scikit-learn==1.5.2",
        "joblib==1.4.2",
        "shap==0.46.0",
        "pytest==8.3.3",
    ]
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *packages],
        check=True,
    )

import joblib
import numpy as np
import pandas as pd
import sklearn

print("Python:", sys.version.split()[0])
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)

## 2. Huấn luyện bằng source code của dự án

In [ ]:
from src.config import DATA_PATH
from src.pipeline import run_pipeline

run = run_pipeline(DATA_PATH)
run

## 3. Xác nhận kết quả trùng với artifact chuẩn

In [ ]:
from src.config import METRICS_PATH
import json

metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
metrics

## 4. Chạy toàn bộ kiểm thử

In [ ]:
test_result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"],
    check=True,
    capture_output=True,
    text=True,
)
print(test_result.stdout)

## 5. Dự báo thử và kiểm tra schema đầu ra

In [ ]:
from src.artifacts.loader import clear_model_cache
from src.serving.predictor import predict_one

clear_model_cache()

sample_property = {
    "Property Type": "Nhà riêng",
    "location_area": "Quận 1",
    "Area": 80.0,
    "Bedrooms": 3,
    "Bathrooms": 2,
    "Floors": 2,
    "Width": 4.0,
    "Length": 20.0,
    "Alley Width": 3.0,
    "Direction": "Đông",
    "Position": "Trong hẻm",
    "car_alley": True,
}

prediction = predict_one(sample_property)
prediction

## Ghi chú

- Mô hình ước lượng **giá đăng tham khảo**, không phải giá giao dịch hoặc chứng thư thẩm định.
- Nếu kiểm tra metrics thất bại, hãy chọn **Runtime → Restart session and run all** để Colab nạp đúng phiên bản thư viện vừa cài.
- Notebook cố ý gọi các hàm trong `src/` thay vì chép lại pipeline, nhờ đó tránh hai phiên bản logic bị lệch nhau.